# RQ2 Colab runner: Semantic ID structure ablations

This notebook runs the controlled RQ2 experiments on Amazon Beauty:

- TIGER original baseline
- TIGER with reversed Semantic-ID token order
- TIGER with permuted Semantic-ID token order
- TIGER with collision/disambiguation token first

The comparison is controlled: same data, same RQ-VAE codes, same training budget, same learning rate, same seed, same decoding setup. Only the RQ2 flag changes.

Codex should edit normal repo files (`src/*.py`, `scripts/`, `report/`). This notebook is only a Colab GPU runner.

In [ ]:
# Repo / runtime config.
REPO_URL = "https://github.com/MayaLager/TIGER-modi.git"
BRANCH = "ksusha/errors-fix"
PROJECT_DIR = "/content/TIGER-modi"

# Persist large artifacts and results across Colab reconnects.
USE_DRIVE_CACHE = True
DRIVE_ROOT = "/content/drive/MyDrive/TIGER-modi-colab"

# Controlled RQ2 protocol.
EPOCHS = 100
LR = 3e-4
LR_TAG = "3e-4"
BATCH_SIZE = 256
NUM_WORKERS = 2
PIN_MEMORY = True
D_MODEL = 128
NUM_LAYERS = 4
NUM_BEAMS = 30
SEED = 42
DEVICE = "cuda"
USE_AMP = True
AMP_DTYPE = "bfloat16"  # A100-friendly; use "float16" only if needed
EVAL_EVERY = 10

# Online monitoring. Set USE_WANDB=False if you do not want to log to W&B.
USE_WANDB = True
WANDB_PROJECT = "tiger-rq2-beauty"
WANDB_ENTITY = None      # e.g. "your-team"; None uses your default account
WANDB_MODE = "online"   # "online", "offline", or "disabled"
WANDB_TAGS = "rq2,beauty,tiger,e100,lr3e-4"
LOG_GRAD_NORM = True
GRAD_NORM_EVERY = 10    # compute grad norm every N train batches

# Do not retrain original TIGER if we already trust an existing baseline run.
# The RQ2 ablations below still use exactly the same budget/hyperparameters.
RUN_BASELINE = False
BASELINE_REFERENCE = {
    "name": "orig_reference_existing_e100_lr3e-4",
    "recall@5": 0.0321,
    "ndcg@5": 0.0208,
    "recall@10": 0.0501,
    "ndcg@10": 0.0266,
}

# If False, completed JSON outputs are not recomputed.
FORCE_RERUN = False

DATA_DIR = "data/processed/Beauty"
CODES = f"{DATA_DIR}/codes_rqvae_L3_K256.npy"
RUN_DIR = "runs/rq2"
LOG_DIR = f"{RUN_DIR}/logs"
METRICS_DIR = f"{RUN_DIR}/metrics"


In [ ]:
# Hardware check. If this does not show a GPU, change the Colab runtime type.
!nvidia-smi || true
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

In [ ]:
# Clone or update the experiment branch in the Colab VM.
import os, subprocess

def sh(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, cwd=cwd, check=True)

if not os.path.exists(PROJECT_DIR):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}")
else:
    sh("git fetch origin", cwd=PROJECT_DIR)
    sh(f"git checkout {BRANCH}", cwd=PROJECT_DIR)
    sh("git pull --ff-only", cwd=PROJECT_DIR)

%cd {PROJECT_DIR}
!git status --short --branch
!git log --oneline -3

In [ ]:
# Mount Drive and symlink generated artifacts there.
# This is what keeps data/codes/runs after the Colab VM is deleted.
import os, pathlib, shutil

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    root = pathlib.Path(DRIVE_ROOT)
    for sub in ["data/raw", "data/processed", "runs"]:
        (root / sub).mkdir(parents=True, exist_ok=True)
    pathlib.Path("data").mkdir(exist_ok=True)
    for local, target in [
        ("data/raw", root / "data/raw"),
        ("data/processed", root / "data/processed"),
        ("runs", root / "runs"),
    ]:
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.exists(local):
            shutil.rmtree(local)
        os.symlink(target, local)

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
!ls -lah data runs


In [ ]:
# Install Colab dependencies without reinstalling PyTorch CUDA.
!python -m pip install -q "sentence-transformers>=2.7" "transformers>=4.40" "scikit-learn>=1.3" "tqdm>=4.66" "wandb>=0.17"

import torch, transformers, sklearn, wandb
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("sklearn", sklearn.__version__)
print("wandb", wandb.__version__)


In [ ]:
# W&B login. Run once per Colab runtime when USE_WANDB=True.
# Paste an API key from https://wandb.ai/authorize when prompted.
if USE_WANDB and WANDB_MODE == "online":
    import os
    from getpass import getpass
    import wandb

    if not os.environ.get("WANDB_API_KEY"):
        secret_key = None
        try:
            from google.colab import userdata
            secret_key = userdata.get("WANDB_API_KEY")
        except Exception:
            secret_key = None
        key = (secret_key or getpass("W&B API key from https://wandb.ai/authorize: ")).strip()
        if not key:
            raise RuntimeError("Empty W&B API key. Set USE_WANDB=False to skip online logging.")
        os.environ["WANDB_API_KEY"] = key

    wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
    print(f"W&B online logging enabled: project={WANDB_PROJECT}")
elif USE_WANDB and WANDB_MODE == "offline":
    import os
    os.environ["WANDB_MODE"] = "offline"
    print("W&B offline logging enabled")
else:
    print("W&B disabled")


In [ ]:
# Small helper: stream command output to notebook and save a log file in Drive.
# We force unbuffered subprocess output, so epoch logs appear online in the cell.
import os, subprocess, time, json, glob


def run_logged(args, out_json=None, log_path=None, force=False):
    if out_json and os.path.exists(out_json) and not force:
        print(f"[skip] {out_json} exists")
        return
    if log_path:
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
    if out_json:
        os.makedirs(os.path.dirname(out_json), exist_ok=True)
    print("$ " + " ".join(map(str, args)))
    t0 = time.time()
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["WANDB_MODE"] = WANDB_MODE
    if USE_WANDB and WANDB_MODE == "online":
        has_env_key = bool(env.get("WANDB_API_KEY"))
        has_netrc_key = os.path.exists(os.path.expanduser("~/.netrc"))
        if not has_env_key and not has_netrc_key:
            raise RuntimeError("W&B online logging is enabled, but no login was found. Run the W&B login cell, or set USE_WANDB=False.")
    proc = subprocess.Popen(
        args,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    with open(log_path, "w") if log_path else open(os.devnull, "w") as log:
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
    rc = proc.wait()
    print(f"[exit {rc}] elapsed={(time.time() - t0) / 60:.1f} min")
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}")


def out_path(name):
    return f"{RUN_DIR}/beauty_tiger_{name}_e{EPOCHS}_lr{LR_TAG}_seed{SEED}.json"


def log_path(name):
    return f"{LOG_DIR}/beauty_tiger_{name}_e{EPOCHS}_lr{LR_TAG}_seed{SEED}.log"


def metrics_path(name):
    return f"{METRICS_DIR}/beauty_tiger_{name}_e{EPOCHS}_lr{LR_TAG}_seed{SEED}.jsonl"


## Prepare data and RQ-VAE codes

Run these cells once per Drive cache. If outputs already exist, cells skip or reuse cached artifacts.

In [ ]:
# Download and preprocess Amazon Beauty 5-core.
if not os.path.exists(f"{DATA_DIR}/meta.json") or FORCE_RERUN:
    run_logged(["python", "src/data/download.py", "--category", "Beauty", "--out", "data/raw", "--review-set", "5core"],
               log_path=f"{LOG_DIR}/download_beauty.log", force=True)
    run_logged(["python", "src/data/preprocess.py", "--category", "Beauty", "--raw", "data/raw", "--out", DATA_DIR, "--kcore", "5"],
               log_path=f"{LOG_DIR}/preprocess_beauty.log", force=True)
else:
    print(f"[skip] {DATA_DIR}/meta.json exists")

!cat data/processed/Beauty/meta.json

In [ ]:
# Build paper-shaped RQ-VAE Semantic IDs if missing.
run_logged([
    "python", "src/train_semantic_ids.py",
    "--data", DATA_DIR,
    "--id-method", "rqvae",
    "--num-levels", "3",
    "--codebook-size", "256",
    "--rqvae-epochs", "200",
    "--device", DEVICE,
    "--seed", str(SEED),
], out_json=f"{DATA_DIR}/codes_rqvae_L3_K256.meta.json", log_path=f"{LOG_DIR}/semantic_ids_rqvae_L3_K256.log", force=FORCE_RERUN)

!ls -lh data/processed/Beauty/codes_rqvae_L3_K256.npy data/processed/Beauty/codes_rqvae_L3_K256.meta.json

## RQ2 controlled runs

Each cell changes exactly one thing. If a run finishes, its JSON and log are stored in Drive under `runs/rq2/`.

In [ ]:
COMMON_TIGER_ARGS = [
    "python", "-u", "src/train_tiger.py",
    "--data", DATA_DIR,
    "--codes", CODES,
    "--codebook-size", "256",
    "--epochs", str(EPOCHS),
    "--lr", str(LR),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--d-model", str(D_MODEL),
    "--num-layers", str(NUM_LAYERS),
    "--num-beams", str(NUM_BEAMS),
    "--eval-every", str(EVAL_EVERY),
    "--device", DEVICE,
    "--seed", str(SEED),
]
if PIN_MEMORY:
    COMMON_TIGER_ARGS += ["--pin-memory"]
if USE_AMP:
    COMMON_TIGER_ARGS += ["--amp", "--amp-dtype", AMP_DTYPE]
if LOG_GRAD_NORM:
    COMMON_TIGER_ARGS += ["--log-grad-norm", "--grad-norm-every", str(GRAD_NORM_EVERY)]


def wandb_args(name):
    if not USE_WANDB:
        return []
    args = [
        "--wandb-project", WANDB_PROJECT,
        "--wandb-run-name", f"rq2-{name}-e{EPOCHS}-lr{LR_TAG}-seed{SEED}",
        "--wandb-mode", WANDB_MODE,
        "--wandb-tags", f"{WANDB_TAGS},{name}",
    ]
    if WANDB_ENTITY:
        args += ["--wandb-entity", WANDB_ENTITY]
    return args


def run_tiger_rq2(name, extra_args):
    run_logged(
        COMMON_TIGER_ARGS
        + extra_args
        + wandb_args(name)
        + ["--metrics-log", metrics_path(name), "--out", out_path(name)],
        out_json=out_path(name),
        log_path=log_path(name),
        force=FORCE_RERUN,
    )


In [ ]:
# Optional progress/status cell.
# Run between long experiment cells, after reconnect, or after an interruption.
import os, json, glob

status_rows = []
for path in sorted(glob.glob(f"{METRICS_DIR}/beauty_tiger_*.jsonl")):
    name = os.path.basename(path).replace("beauty_tiger_", "").split(f"_e{EPOCHS}_")[0]
    last_epoch = None
    best_val = None
    test = None
    last_loss = None
    for line in open(path):
        rec = json.loads(line)
        if rec.get("phase") == "epoch":
            last_epoch = rec.get("epoch")
            last_loss = rec.get("loss")
            if "best_val_recall@10" in rec:
                best_val = rec["best_val_recall@10"]
        elif rec.get("phase") == "test":
            test = rec.get("test", {})
    status_rows.append({
        "name": name,
        "last_epoch": last_epoch,
        "last_loss": last_loss,
        "best_val_R@10": best_val,
        "test_R@10": None if test is None else test.get("recall@10"),
        "test_N@10": None if test is None else test.get("ndcg@10"),
    })

if not status_rows:
    print("No metrics JSONL files yet.")
else:
    try:
        import pandas as pd
        display(pd.DataFrame(status_rows))
    except Exception:
        for row in status_rows:
            print(row)


In [ ]:
# Local plots from JSONL metrics, useful if W&B is offline or after reconnect.
import os, json, glob

records = []
for path in sorted(glob.glob(f"{METRICS_DIR}/beauty_tiger_*.jsonl")):
    name = os.path.basename(path).replace("beauty_tiger_", "").split(f"_e{EPOCHS}_")[0]
    for line in open(path):
        rec = json.loads(line)
        if rec.get("phase") == "epoch":
            row = {"name": name, "epoch": rec.get("epoch"), "loss": rec.get("loss")}
            row["grad_norm_mean"] = rec.get("grad_norm_mean")
            row["grad_norm_max"] = rec.get("grad_norm_max")
            val = rec.get("val") or {}
            for key in ("recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"):
                row[f"val_{key}"] = val.get(key)
            records.append(row)

if not records:
    print("No epoch metrics yet.")
else:
    import pandas as pd
    import matplotlib.pyplot as plt
    df = pd.DataFrame(records)
    display(df.tail(20))
    for metric in ["loss", "grad_norm_mean", "val_recall@10", "val_ndcg@10"]:
        if metric not in df or df[metric].dropna().empty:
            continue
        plt.figure(figsize=(8, 4))
        for name, part in df.groupby("name"):
            plt.plot(part["epoch"], part[metric], marker="o", label=name)
        plt.title(metric)
        plt.xlabel("epoch")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()


In [ ]:
# Baseline: original TIGER, no RQ2 modification.
# Usually keep RUN_BASELINE=False and compare against BASELINE_REFERENCE.
if RUN_BASELINE:
    run_tiger_rq2("orig", [])
else:
    print("[skip] RUN_BASELINE=False")
    print("baseline reference:", BASELINE_REFERENCE)


In [ ]:
# RQ2: reverse content-token order.
run_tiger_rq2("reversed", ["--token-order", "reversed"])

In [ ]:
# RQ2: deterministic random content-token permutation (seed-controlled).
run_tiger_rq2("permuted", ["--token-order", "permuted"])

In [ ]:
# RQ2: put collision/disambiguation token first.
run_tiger_rq2("collision_first", ["--collision-first"])

## Summarize RQ2 results

This cell writes both CSV and Markdown summary files into Drive-backed `runs/rq2/`.

In [ ]:
import json, os, glob

names = ["orig", "reversed", "permuted", "collision_first"]
rows = []
baseline = None
for name in names:
    path = out_path(name)
    if not os.path.exists(path):
        print(f"[missing] {path}")
        continue
    d = json.load(open(path))
    test = d["test"]
    row = {
        "name": name,
        "path": path,
        "best_val_recall@10": d.get("best_val_recall@10"),
        "recall@5": test.get("recall@5"),
        "ndcg@5": test.get("ndcg@5"),
        "recall@10": test.get("recall@10"),
        "ndcg@10": test.get("ndcg@10"),
        "recall@20": test.get("recall@20"),
        "ndcg@20": test.get("ndcg@20"),
        "invalid_code_rate": test.get("invalid_code_rate"),
        "invalid_code_rate@10": test.get("invalid_code_rate@10"),
        "collision_rate_pre_disambiguation": test.get("collision_rate_pre_disambiguation"),
        "mean_valid_candidates": test.get("mean_valid_candidates"),
        "decode_latency_s": test.get("decode_latency_s"),
    }
    rows.append(row)
    if name == "orig":
        baseline = row

if baseline is None and BASELINE_REFERENCE:
    baseline = {
        "name": BASELINE_REFERENCE["name"],
        "path": "manual_reference",
        "best_val_recall@10": None,
        "recall@5": BASELINE_REFERENCE.get("recall@5"),
        "ndcg@5": BASELINE_REFERENCE.get("ndcg@5"),
        "recall@10": BASELINE_REFERENCE.get("recall@10"),
        "ndcg@10": BASELINE_REFERENCE.get("ndcg@10"),
        "recall@20": None,
        "ndcg@20": None,
        "invalid_code_rate": None,
        "invalid_code_rate@10": None,
        "collision_rate_pre_disambiguation": None,
        "mean_valid_candidates": None,
        "decode_latency_s": None,
    }
    rows = [baseline] + rows

if baseline:
    for row in rows:
        row["delta_recall@10_vs_orig"] = (
            None if row.get("recall@10") is None or baseline.get("recall@10") is None
            else row["recall@10"] - baseline["recall@10"]
        )
        row["delta_ndcg@10_vs_orig"] = (
            None if row.get("ndcg@10") is None or baseline.get("ndcg@10") is None
            else row["ndcg@10"] - baseline["ndcg@10"]
        )

summary_csv = f"{RUN_DIR}/summary_rq2_e{EPOCHS}_lr{LR_TAG}_seed{SEED}.csv"
summary_md = f"{RUN_DIR}/summary_rq2_e{EPOCHS}_lr{LR_TAG}_seed{SEED}.md"

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
    df.to_csv(summary_csv, index=False)
    md = df.to_markdown(index=False)
except Exception:
    import csv
    keys = list(rows[0]) if rows else []
    with open(summary_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader(); writer.writerows(rows)
    md_lines = ["| " + " | ".join(keys) + " |", "| " + " | ".join(["---"] * len(keys)) + " |"]
    for row in rows:
        md_lines.append("| " + " | ".join(str(row.get(k, "")) for k in keys) + " |")
    md = "\n".join(md_lines)

with open(summary_md, "w") as f:
    f.write(md + "\n")

print("saved", summary_csv)
print("saved", summary_md)
print(md)


## Shutdown reminder

After the last run is saved, disconnect the Colab GPU runtime to stop spending compute units:

- Browser Colab: `Runtime -> Disconnect and delete runtime`
- VS Code Colab extension: Command Palette -> `Colab: Remove Server`